# FairLens Risk V2 Colab Training (HF + LightGBM)

End-to-end notebook for manual Colab training:
- pull datasets directly from Hugging Face
- clean + map to `risk-v2.0.0` features
- train CatBoost + LightGBM challenger
- calibrate + select threshold with downside-protection objective
- export artifacts in current `ml-service`-compatible names


In [1]:
!pip -q install -U pip
!pip -q install pandas numpy scikit-learn catboost lightgbm pyarrow joblib huggingface_hub


In [2]:
from __future__ import annotations

import json
import zipfile
from pathlib import Path
from typing import Iterable

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from huggingface_hub import HfApi, hf_hub_download
from lightgbm import LGBMClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, brier_score_loss, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

RISK_SCHEMA_VERSION = 'risk-v2.0.0'
TARGET_COLUMN = 'default_90d_proxy'
MODEL_VERSION = 'ensemble-catboost-lgbm-v2-colab-hf'

FEATURE_COLUMNS = [
    'segment_code',
    'monthly_inflow',
    'monthly_outflow',
    'inflow_volatility_90d',
    'outflow_volatility_90d',
    'deposit_count_30d',
    'days_since_last_income',
    'avg_balance_30d',
    'min_balance_30d',
    'negative_balance_days_30d',
    'essential_spend_ratio',
    'active_loan_count',
    'monthly_installment_burden',
    'purchase_amount',
    'tenure_weeks',
    'purchase_to_inflow_ratio',
    'installment_to_inflow_ratio',
    'total_burden_ratio',
    'buffer_ratio',
    'stress_index',
]

REASON_CODE_CATALOG = [
    'HIGH_TOTAL_BURDEN',
    'HIGH_STRESS_INDEX',
    'NEGATIVE_BALANCE_FREQUENCY',
    'INFLOW_VOLATILITY_PRESSURE',
    'CASHFLOW_LOAD',
    'BUFFER_STRENGTH',
    'INCOME_REGULARITY',
]

SEGMENT_CODE_MAP = {
    'student': 0,
    'gig_worker': 1,
    'informal_worker': 2,
    'salaried': 3,
    'self_employed': 4,
    'unknown': 5,
}

# Update these repo ids if you prefer different HF mirrors.
DATASET_SOURCES = {
    'home_credit': {
        'repo_id': 'jlh/home-credit',
        'candidates': ['application_train.csv', 'application_train.parquet', 'train.csv', 'data.csv'],
    },
    'gmsc': {
        'repo_id': 'Keyurjotaniya007/Give-Me-Some-Credit',
        'candidates': ['cs-training.csv', 'train.csv', 'data.csv'],
    },
    'heloc': {
        'repo_id': 'mstz/heloc',
        'candidates': ['heloc_dataset.csv', 'heloc.csv', 'train.csv', 'data.csv'],
    },
}

HF_TOKEN = None  # set if dataset is private/gated
ARTIFACTS_DIR = Path('/content/fairlens_artifacts/risk_v2')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def _to_numeric(series: pd.Series, default: float = 0.0) -> pd.Series:
    return pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(default)


def _clamp(series: pd.Series, lower: float = 0.0, upper: float = 1.0) -> pd.Series:
    return series.clip(lower=lower, upper=upper)


def _safe_divide(numerator: pd.Series, denominator: pd.Series, fallback: float = 0.0) -> pd.Series:
    den = denominator.replace(0, np.nan)
    out = numerator / den
    return out.replace([np.inf, -np.inf], np.nan).fillna(fallback)


def _match_candidate(files: Iterable[str], candidates: list[str]) -> str:
    file_list = list(files)
    lower_lookup = {f.lower(): f for f in file_list}
    for name in candidates:
        if name.lower() in lower_lookup:
            return lower_lookup[name.lower()]
    for c in candidates:
        needle = c.lower().replace('.csv', '').replace('.parquet', '')
        for f in file_list:
            base = Path(f).name.lower()
            if needle in base and (base.endswith('.csv') or base.endswith('.parquet')):
                return f
    raise FileNotFoundError(f'No matching dataset file found. candidates={candidates}')


def download_from_hf(repo_id: str, candidates: list[str], token: str | None = None) -> Path:
    api = HfApi()
    files = api.list_repo_files(repo_id=repo_id, repo_type='dataset', token=token)
    filename = _match_candidate(files, candidates)
    local_path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type='dataset', token=token)
    print(f'Downloaded: {repo_id}/{filename}')
    return Path(local_path)


def _read_any(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path)


def _enforce_informal_min_share(frame: pd.DataFrame, min_share: float = 0.05) -> pd.DataFrame:
    target_count = int(len(frame) * min_share)
    current_count = int((frame['segment'] == 'informal_worker').sum())
    if current_count >= target_count:
        return frame

    needed = target_count - current_count
    if needed <= 0:
        return frame

    candidate_mask = ~frame['segment'].isin(['student', 'gig_worker'])
    candidates = frame[candidate_mask].copy()
    if candidates.empty:
        return frame

    candidates = candidates.assign(
        informal_score=(1 - _clamp(candidates['buffer_ratio'], 0, 1)) * 0.5
        + _clamp(candidates['inflow_volatility_90d'], 0, 1) * 0.3
        + (1 - _clamp(candidates['deposit_count_30d'] / 10.0, 0, 1)) * 0.2
    )
    selected_idx = candidates.sort_values('informal_score', ascending=False).head(needed).index
    frame.loc[selected_idx, 'segment'] = 'informal_worker'
    return frame


def assign_segments(frame: pd.DataFrame, age_years: pd.Series | None = None) -> pd.DataFrame:
    q20_income = frame['monthly_inflow'].quantile(0.20)
    q35_income = frame['monthly_inflow'].quantile(0.35)
    q30_purchase = frame['purchase_amount'].quantile(0.30)

    age = age_years if age_years is not None else pd.Series(99, index=frame.index)
    segment = pd.Series('self_employed', index=frame.index, dtype='object')

    salaried_mask = (frame['inflow_volatility_90d'] <= 0.18) & (frame['deposit_count_30d'] >= 3)
    gig_mask = (frame['inflow_volatility_90d'] >= 0.35) & (frame['deposit_count_30d'] >= 4)
    informal_mask = (
        (frame['deposit_count_30d'] <= 3)
        & (frame['monthly_inflow'] <= q35_income)
        & (frame['buffer_ratio'] < 0.18)
    )
    student_mask = (age <= 24) | ((frame['monthly_inflow'] <= q20_income) & (frame['purchase_amount'] <= q30_purchase))

    segment.loc[salaried_mask] = 'salaried'
    segment.loc[informal_mask] = 'informal_worker'
    segment.loc[gig_mask] = 'gig_worker'
    segment.loc[student_mask] = 'student'

    frame['segment'] = segment
    frame = _enforce_informal_min_share(frame, min_share=0.05)
    frame['segment_code'] = frame['segment'].map(SEGMENT_CODE_MAP).fillna(SEGMENT_CODE_MAP['unknown']).astype(int)
    return frame


In [4]:
def load_home_credit(path: Path) -> pd.DataFrame:
    df = _read_any(path)
    monthly_income = _to_numeric(df.get('AMT_INCOME_TOTAL', pd.Series(0.0, index=df.index)), 0.0)
    loan_amount = _to_numeric(df.get('AMT_CREDIT', pd.Series(0.0, index=df.index)), 0.0)
    annuity = _to_numeric(df.get('AMT_ANNUITY', pd.Series(0.0, index=df.index)), 0.0)
    days_birth = _to_numeric(df.get('DAYS_BIRTH', pd.Series(-12000, index=df.index)), -12000)
    age_years = (days_birth.abs() / 365.25).clip(lower=16, upper=90)

    monthly_outflow = (monthly_income * 0.58).clip(lower=0)
    out = pd.DataFrame({
        'monthly_inflow': monthly_income,
        'monthly_outflow': monthly_outflow,
        'inflow_volatility_90d': 0.28,
        'outflow_volatility_90d': 0.32,
        'deposit_count_30d': 4,
        'days_since_last_income': 7,
        'avg_balance_30d': (monthly_income - monthly_outflow).clip(lower=0) * 0.4,
        'min_balance_30d': (monthly_income * 0.12).clip(lower=0),
        'negative_balance_days_30d': 2,
        'essential_spend_ratio': 0.62,
        'active_loan_count': 1,
        'monthly_installment_burden': annuity,
        'purchase_amount': loan_amount,
        'tenure_weeks': 24,
    })
    out['purchase_to_inflow_ratio'] = _clamp(_safe_divide(out['purchase_amount'], out['monthly_inflow']), 0, 3)
    out['installment_to_inflow_ratio'] = _clamp(_safe_divide(out['monthly_installment_burden'], out['monthly_inflow']), 0, 2)
    out['total_burden_ratio'] = _clamp(_safe_divide(out['monthly_outflow'] + out['monthly_installment_burden'], out['monthly_inflow']), 0, 2)
    out['buffer_ratio'] = _clamp(_safe_divide(out['min_balance_30d'], out['monthly_inflow']), 0, 3)
    out['stress_index'] = _clamp(0.45 * out['inflow_volatility_90d'] + 0.55 * out['total_burden_ratio'], 0, 1)
    out[TARGET_COLUMN] = _to_numeric(df.get('TARGET', pd.Series(0, index=df.index)), 0).astype(int)
    out['source_dataset'] = 'home_credit'
    return assign_segments(out, age_years=age_years)


def load_gmsc(path: Path) -> pd.DataFrame:
    df = _read_any(path)
    income = _to_numeric(df.get('MonthlyIncome', pd.Series(0.0, index=df.index)), 0.0)
    debt_ratio = _to_numeric(df.get('DebtRatio', pd.Series(0.5, index=df.index)), 0.5)
    revolving = _to_numeric(df.get('RevolvingUtilizationOfUnsecuredLines', pd.Series(0.4, index=df.index)), 0.4)
    late_count = _to_numeric(df.get('NumberOfTimes90DaysLate', pd.Series(0.0, index=df.index)), 0.0)
    age_years = _to_numeric(df.get('age', pd.Series(35.0, index=df.index)), 35.0)

    outflow = (income * _clamp(debt_ratio, 0, 2)).clip(lower=0)
    installment = (outflow * 0.28).clip(lower=0)
    purchase_amount = (income * 0.35).clip(lower=0)
    min_balance = (income * _clamp(1 - debt_ratio, 0, 1) * 0.2).clip(lower=0)

    out = pd.DataFrame({
        'monthly_inflow': income,
        'monthly_outflow': outflow,
        'inflow_volatility_90d': _clamp(revolving, 0, 1),
        'outflow_volatility_90d': _clamp(revolving * 0.8 + 0.1, 0, 1),
        'deposit_count_30d': 4,
        'days_since_last_income': 6,
        'avg_balance_30d': (income - outflow).clip(lower=0) * 0.4,
        'min_balance_30d': min_balance,
        'negative_balance_days_30d': _clamp(late_count, 0, 30),
        'essential_spend_ratio': _clamp(debt_ratio, 0, 2),
        'active_loan_count': _to_numeric(df.get('NumberOfOpenCreditLinesAndLoans', pd.Series(1, index=df.index)), 1).clip(lower=0),
        'monthly_installment_burden': installment,
        'purchase_amount': purchase_amount,
        'tenure_weeks': 24,
    })
    out['purchase_to_inflow_ratio'] = _clamp(_safe_divide(out['purchase_amount'], out['monthly_inflow']), 0, 3)
    out['installment_to_inflow_ratio'] = _clamp(_safe_divide(out['monthly_installment_burden'], out['monthly_inflow']), 0, 2)
    out['total_burden_ratio'] = _clamp(_safe_divide(out['monthly_outflow'] + out['monthly_installment_burden'], out['monthly_inflow']), 0, 2)
    out['buffer_ratio'] = _clamp(_safe_divide(out['min_balance_30d'], out['monthly_inflow']), 0, 3)
    out['stress_index'] = _clamp(0.45 * out['inflow_volatility_90d'] + 0.55 * out['total_burden_ratio'], 0, 1)
    out[TARGET_COLUMN] = _to_numeric(df.get('SeriousDlqin2yrs', pd.Series(0, index=df.index)), 0).astype(int)
    out['source_dataset'] = 'give_me_some_credit'
    return assign_segments(out, age_years=age_years)


def load_heloc(path: Path) -> pd.DataFrame:
    df = _read_any(path)
    risk_performance = df.get('RiskPerformance', pd.Series('Bad', index=df.index)).astype(str).str.lower().str.strip()
    bad_target = risk_performance.map({'bad': 1, 'good': 0}).fillna(1).astype(int)

    ext_risk = _to_numeric(df.get('ExternalRiskEstimate', pd.Series(60, index=df.index)), 60)
    msince = _to_numeric(df.get('MSinceMostRecentTradeOpen', pd.Series(36, index=df.index)), 36)
    trades = _to_numeric(df.get('NumTotalTrades', pd.Series(10, index=df.index)), 10)

    monthly_inflow = (ext_risk * 800).clip(lower=5000)
    monthly_outflow = monthly_inflow * 0.62
    installment = monthly_inflow * 0.12
    purchase_amount = monthly_inflow * 0.32

    out = pd.DataFrame({
        'monthly_inflow': monthly_inflow,
        'monthly_outflow': monthly_outflow,
        'inflow_volatility_90d': _clamp(1 - (ext_risk / 100), 0, 1),
        'outflow_volatility_90d': _clamp(0.2 + (msince / 200), 0, 1),
        'deposit_count_30d': 4,
        'days_since_last_income': _clamp(msince / 30, 0, 30),
        'avg_balance_30d': (monthly_inflow - monthly_outflow).clip(lower=0) * 0.35,
        'min_balance_30d': monthly_inflow * 0.10,
        'negative_balance_days_30d': _clamp(trades / 2.5, 0, 30),
        'essential_spend_ratio': 0.62,
        'active_loan_count': trades.clip(lower=0),
        'monthly_installment_burden': installment,
        'purchase_amount': purchase_amount,
        'tenure_weeks': 24,
    })
    out['purchase_to_inflow_ratio'] = _clamp(_safe_divide(out['purchase_amount'], out['monthly_inflow']), 0, 3)
    out['installment_to_inflow_ratio'] = _clamp(_safe_divide(out['monthly_installment_burden'], out['monthly_inflow']), 0, 2)
    out['total_burden_ratio'] = _clamp(_safe_divide(out['monthly_outflow'] + out['monthly_installment_burden'], out['monthly_inflow']), 0, 2)
    out['buffer_ratio'] = _clamp(_safe_divide(out['min_balance_30d'], out['monthly_inflow']), 0, 3)
    out['stress_index'] = _clamp(0.45 * out['inflow_volatility_90d'] + 0.55 * out['total_burden_ratio'], 0, 1)
    out[TARGET_COLUMN] = bad_target
    out['source_dataset'] = 'heloc'
    return assign_segments(out)


In [5]:
home_credit_file = download_from_hf(DATASET_SOURCES['home_credit']['repo_id'], DATASET_SOURCES['home_credit']['candidates'], HF_TOKEN)
gmsc_file = download_from_hf(DATASET_SOURCES['gmsc']['repo_id'], DATASET_SOURCES['gmsc']['candidates'], HF_TOKEN)
heloc_file = download_from_hf(DATASET_SOURCES['heloc']['repo_id'], DATASET_SOURCES['heloc']['candidates'], HF_TOKEN)

dataset = pd.concat([
    load_home_credit(home_credit_file),
    load_gmsc(gmsc_file),
    load_heloc(heloc_file),
], ignore_index=True)

dataset = dataset.replace([np.inf, -np.inf], np.nan).fillna(0)
dataset[TARGET_COLUMN] = dataset[TARGET_COLUMN].astype(int)
for col in FEATURE_COLUMNS:
    dataset[col] = _to_numeric(dataset[col], 0)

print('Rows:', len(dataset))
print('Default rate:', round(float(dataset[TARGET_COLUMN].mean()), 4))
print('Source distribution:')
print(dataset['source_dataset'].value_counts(dropna=False))
print('Segment distribution:')
print(dataset['segment'].value_counts(dropna=False))


Downloaded: jlh/home-credit/data/train-00000-of-00001-e68d01965482ae18.parquet
Downloaded: Keyurjotaniya007/Give-Me-Some-Credit/train.csv
Downloaded: mstz/heloc/risk/train.csv
Rows: 467970
Default rate: 0.0968
Source distribution:
source_dataset
home_credit            307511
give_me_some_credit    150000
heloc                   10459
Name: count, dtype: int64
Segment distribution:
segment
self_employed      260300
student             84132
salaried            57857
gig_worker          42806
informal_worker     22875
Name: count, dtype: int64


In [6]:
X = dataset[FEATURE_COLUMNS].copy()
y = dataset[TARGET_COLUMN].astype(int).copy()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

segment_train = dataset.loc[X_train.index, 'segment'].astype(str)
segment_valid = dataset.loc[X_valid.index, 'segment'].astype(str)

# Shared segment-aware weights for both models (comparable fairness treatment).
sample_weight = np.ones(len(X_train), dtype=float)
sample_weight[y_train.to_numpy() == 1] *= 1.8
sample_weight[segment_train.to_numpy() == 'informal_worker'] *= 2.5
sample_weight[(segment_train.to_numpy() == 'informal_worker') & (y_train.to_numpy() == 1)] *= 1.6
sample_weight[segment_train.to_numpy() == 'gig_worker'] *= 1.15

cat_model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    depth=8,
    learning_rate=0.05,
    iterations=700,
    l2_leaf_reg=8.0,
    random_seed=42,
    verbose=False,
)
cat_model.fit(X_train, y_train, sample_weight=sample_weight, eval_set=(X_valid, y_valid), verbose=False)

lgbm_model = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=42,
)
lgbm_model.fit(X_train, y_train, sample_weight=sample_weight)

print('Train segment counts:')
print(segment_train.value_counts(dropna=False))
print('Avg sample weight by segment:')
print(pd.DataFrame({'segment': segment_train, 'w': sample_weight}).groupby('segment')['w'].mean().sort_values(ascending=False))
print('Train:', X_train.shape, 'Valid:', X_valid.shape, 'Test:', X_test.shape)


[LightGBM] [Info] Number of positive: 28998, number of negative: 270502
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.087797 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3639
[LightGBM] [Info] Number of data points in the train set: 299500, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.166856 -> initscore=-1.608078
[LightGBM] [Info] Start training from score -1.608078
Train segment counts:
segment
self_employed      166423
student             53905
salaried            37208
gig_worker          27374
informal_worker     14590
Name: count, dtype: int64
Avg sample weight by segment:
segment
informal_worker    2.837923
gig_worker         1.284770
student            1.149136
self_employed      1.063020
salaried           1.016771
Name: w, dtype: float64
Train: (299500, 20) Valid: (74876, 20) Test: (93594, 20)


In [7]:
def evaluate_probs(name: str, labels: np.ndarray, probs: np.ndarray) -> dict:
    return {
        'model': name,
        'roc_auc': float(roc_auc_score(labels, probs)),
        'pr_auc': float(average_precision_score(labels, probs)),
        'brier': float(brier_score_loss(labels, probs)),
    }


cat_valid_raw = cat_model.predict_proba(X_valid)[:, 1]
lgbm_valid_raw = lgbm_model.predict_proba(X_valid)[:, 1]

cat_calibrator = IsotonicRegression(out_of_bounds='clip').fit(cat_valid_raw, y_valid.to_numpy())
lgbm_calibrator = IsotonicRegression(out_of_bounds='clip').fit(lgbm_valid_raw, y_valid.to_numpy())

cat_valid = cat_calibrator.predict(cat_valid_raw)
lgbm_valid = lgbm_calibrator.predict(lgbm_valid_raw)

baseline_threshold = 0.55
baseline_recall_bad = float(recall_score(y_valid, (cat_valid >= baseline_threshold).astype(int), zero_division=0))
baseline_approval = float(np.mean(cat_valid < baseline_threshold))
min_approval = max(0.0, baseline_approval - 0.10)
min_approval_floor = max(min_approval, 0.85)
min_informal_recall = 0.45

candidates = [('catboost_only', 1.0), ('lightgbm_only', 0.0)] + [(f'ensemble_{w:.2f}', float(w)) for w in np.linspace(0.50, 0.90, 9)]
best = {
    'name': 'catboost_only',
    'weight_cat': 1.0,
    'threshold': 0.55,
    'recall_bad': -1.0,
    'approval_rate': 0.0,
    'validation_recall_gap': 1.0,
    'validation_informal_recall': 0.0,
    'selection_score': -999.0,
}

# Fairness-constrained objective: maximize recall while penalizing segment recall gaps.
# selection_score = recall_bad - 0.45 * validation_recall_gap
# hard constraints: approval_rate >= min_approval_floor and informal_worker recall >= min_informal_recall
for name, w_cat in candidates:
    blended = w_cat * cat_valid + (1 - w_cat) * lgbm_valid
    for threshold in np.linspace(0.05, 0.80, 76):
        preds = (blended >= threshold).astype(int)
        recall_bad = float(recall_score(y_valid, preds, zero_division=0))
        approval_rate = float(np.mean(blended < threshold))
        seg_recalls = []
        seg_recall_map = {}
        for seg in ['student', 'gig_worker', 'informal_worker']:
            seg_mask = (segment_valid.to_numpy() == seg)
            if int(seg_mask.sum()) >= 200:
                seg_recall = float(recall_score(y_valid[seg_mask], preds[seg_mask], zero_division=0))
                seg_recalls.append(seg_recall)
                seg_recall_map[seg] = seg_recall
        recall_gap = (max(seg_recalls) - min(seg_recalls)) if len(seg_recalls) >= 2 else 0.0
        informal_recall = seg_recall_map.get('informal_worker', 0.0)
        selection_score = recall_bad - 0.45 * recall_gap
        constraints_ok = (approval_rate >= min_approval_floor) and (informal_recall >= min_informal_recall)
        if constraints_ok and selection_score > best['selection_score']:
            best = {
                'name': name,
                'weight_cat': float(w_cat),
                'threshold': float(threshold),
                'recall_bad': recall_bad,
                'approval_rate': approval_rate,
                'validation_recall_gap': float(recall_gap),
                'validation_informal_recall': float(informal_recall),
                'selection_score': float(selection_score),
            }

cat_test = cat_calibrator.predict(cat_model.predict_proba(X_test)[:, 1])
lgbm_test = lgbm_calibrator.predict(lgbm_model.predict_proba(X_test)[:, 1])
ensemble_test = best['weight_cat'] * cat_test + (1 - best['weight_cat']) * lgbm_test

metrics = {
    'catboost': evaluate_probs('catboost', y_test.to_numpy(), cat_test),
    'lightgbm': evaluate_probs('lightgbm', y_test.to_numpy(), lgbm_test),
    'selected_blend': evaluate_probs(best['name'], y_test.to_numpy(), ensemble_test),
    'baseline_valid': {
        'recall_bad_at_0_55': baseline_recall_bad,
        'approval_rate_at_0_55': baseline_approval,
    },
    'selected': best,
}

test_segments = dataset.loc[X_test.index, 'segment'].astype(str)
fairness = {}
for seg in ['student', 'gig_worker', 'informal_worker']:
    mask = test_segments == seg
    n = int(mask.sum())
    if n == 0:
        fairness[seg] = {'n': 0, 'recall_bad': None, 'approval_rate': None}
        continue
    y_seg = y_test[mask]
    risk_seg = ensemble_test[mask]
    preds_seg = (risk_seg >= best['threshold']).astype(int)
    fairness[seg] = {
        'n': n,
        'recall_bad': float(recall_score(y_seg, preds_seg, zero_division=0)),
        'approval_rate': float(np.mean(risk_seg < best['threshold'])),
    }

print(json.dumps(metrics, indent=2))
print(json.dumps(fairness, indent=2))


{
  "catboost": {
    "model": "catboost",
    "roc_auc": 0.7656556935187716,
    "pr_auc": 0.46078307744173597,
    "brier": 0.06484805171262377
  },
  "lightgbm": {
    "model": "lightgbm",
    "roc_auc": 0.7699453797690767,
    "pr_auc": 0.46769368913277987,
    "brier": 0.06472339899340299
  },
  "selected_blend": {
    "model": "catboost_only",
    "roc_auc": 0.7656556935187716,
    "pr_auc": 0.46078307744173597,
    "brier": 0.06484805171262377
  },
  "baseline_valid": {
    "recall_bad_at_0_55": 0.256,
    "approval_rate_at_0_55": 0.9739569421443454
  },
  "selected": {
    "name": "catboost_only",
    "weight_cat": 1.0,
    "threshold": 0.55,
    "recall_bad": -1.0,
    "approval_rate": 0.0,
    "validation_recall_gap": 1.0,
    "validation_informal_recall": 0.0,
    "selection_score": -999.0
  }
}
{
  "student": {
    "n": 16775,
    "recall_bad": 0.6746259153135944,
    "approval_rate": 0.872548435171386
  },
  "gig_worker": {
    "n": 8593,
    "recall_bad": 0.13236481033091

In [8]:
cat_model_path = ARTIFACTS_DIR / 'catboost_model.cbm'
secondary_model_path = ARTIFACTS_DIR / 'ft_model.pkl'  # runtime-compatible name
cat_calibrator_path = ARTIFACTS_DIR / 'cat_calibrator.pkl'
secondary_calibrator_path = ARTIFACTS_DIR / 'ft_calibrator.pkl'  # runtime-compatible name
metadata_path = ARTIFACTS_DIR / 'model_metadata.json'
manifest_path = ARTIFACTS_DIR / 'training_manifest.json'
zip_path = ARTIFACTS_DIR / 'risk_v2_artifacts.zip'

cat_model.save_model(cat_model_path)
joblib.dump(lgbm_model, secondary_model_path)
joblib.dump(cat_calibrator, cat_calibrator_path)
joblib.dump(lgbm_calibrator, secondary_calibrator_path)

metadata = {
    'threshold': best['threshold'],
    'feature_columns': FEATURE_COLUMNS,
    'required_features': FEATURE_COLUMNS,
    'model_version': MODEL_VERSION,
    'schema_version': RISK_SCHEMA_VERSION,
    'reason_code_catalog': REASON_CODE_CATALOG,
    'ensemble': {
        'weight_catboost': best['weight_cat'],
        'weight_ft': round(1 - best['weight_cat'], 6),
    },
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

manifest = {
    'dataset_rows': int(len(dataset)),
    'target': TARGET_COLUMN,
    'schema_version': RISK_SCHEMA_VERSION,
    'model_version': MODEL_VERSION,
    'metrics': metrics,
    'fairness': fairness,
    'artifacts': {
        'catboost_model': cat_model_path.name,
        'ft_model': secondary_model_path.name,
        'cat_calibrator': cat_calibrator_path.name,
        'ft_calibrator': secondary_calibrator_path.name,
        'metadata': metadata_path.name,
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(cat_model_path, arcname=cat_model_path.name)
    zf.write(secondary_model_path, arcname=secondary_model_path.name)
    zf.write(cat_calibrator_path, arcname=cat_calibrator_path.name)
    zf.write(secondary_calibrator_path, arcname=secondary_calibrator_path.name)
    zf.write(metadata_path, arcname=metadata_path.name)
    zf.write(manifest_path, arcname=manifest_path.name)

print('Saved artifacts in:', ARTIFACTS_DIR)
print('Zip bundle:', zip_path)


Saved artifacts in: \content\fairlens_artifacts\risk_v2
Zip bundle: \content\fairlens_artifacts\risk_v2\risk_v2_artifacts.zip


## Notes

- This notebook now uses **LightGBM** as the challenger model.
- Artifact filenames remain unchanged for current `ml-service` loader compatibility.
- If HF dataset filenames differ, update `DATASET_SOURCES[candidates]`.
